In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

if 'HADOOP_HOME' in os.environ:
    os.environ['PATH'] = os.environ['HADOOP_HOME'] + '\\\\bin;' + os.environ.get('PATH', '')

import pyspark
from pyspark.sql import SparkSession


In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [3]:
df = spark.read.parquet('E:\DE-Zoomcamp\PhanDE2026\Homework_06\yellow_tripdata_2025-11.parquet')

In [4]:
df.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [5]:
df = df.repartition(4)

In [6]:
df.write.parquet('yellow_trips_partitioned/', mode='overwrite')

In [9]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [10]:
from pyspark.sql import functions as F

In [13]:
df_new = df \
    .withColumn('pickup_date', F.to_date(df.tpep_pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.tpep_dropoff_datetime))

In [18]:
df_new.createOrReplaceTempView('yellow_taxi')

In [21]:
spark.sql("""
SELECT Count(1)
FROM
    yellow_taxi
WHERE pickup_date = '2025-11-15'
""").show()

+--------+
|count(1)|
+--------+
|  162604|
+--------+



In [22]:
df_with_duration = df.withColumn(
    "duration_hours", 
    (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 3600
)

In [23]:
df_with_duration.select(F.max("duration_hours")).collect()[0][0]

90.64666666666666

In [24]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-03-10 03:40:21--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 2600:9000:21c8:600:b:20a5:b140:21, 2600:9000:21c8:b200:b:20a5:b140:21, 2600:9000:21c8:e800:b:20a5:b140:21, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|2600:9000:21c8:600:b:20a5:b140:21|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: 'taxi_zone_lookup.csv'

     0K .......... ..                                         100% 1.25G=0s

2026-03-10 03:40:22 (1.25 GB/s) - 'taxi_zone_lookup.csv' saved [12331/12331]



In [26]:
df_location = spark.sql("""
SELECT PULocationID, Count(1) AS frequent_pickup
FROM
    yellow_taxi
GROUP BY PULocationID
""")

In [30]:
df_location.orderBy('frequent_pickup', ascending=False).show()

+------------+---------------+
|PULocationID|frequent_pickup|
+------------+---------------+
|         237|         194593|
|         161|         182907|
|         236|         169621|
|         132|         165987|
|         186|         130699|
|         162|         128230|
|         230|         125716|
|         142|         124627|
|         239|         109906|
|         163|         108286|
|         234|         108090|
|         138|         107700|
|          79|         107413|
|         170|         106723|
|          68|         104306|
|          48|          96871|
|         141|          90710|
|         249|          88121|
|         164|          87565|
|         107|          80670|
+------------+---------------+
only showing top 20 rows


In [36]:
df_zones = spark.read.option("header", "true").csv('taxi_zone_lookup.csv')

In [37]:
df_zones.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [39]:
df_location = df_location.withColumn("LocationID", F.col("PULocationID"))

In [45]:
df_join = df_location.join(df_zones, on=['LocationID'], how='inner')

In [46]:
df_join.orderBy('frequent_pickup').show()

+----------+------------+---------------+-------------+--------------------+------------+
|LocationID|PULocationID|frequent_pickup|      Borough|                Zone|service_zone|
+----------+------------+---------------+-------------+--------------------+------------+
|        84|          84|              1|Staten Island|Eltingville/Annad...|   Boro Zone|
|       105|         105|              1|    Manhattan|Governor's Island...| Yellow Zone|
|         5|           5|              1|Staten Island|       Arden Heights|   Boro Zone|
|       187|         187|              3|Staten Island|       Port Richmond|   Boro Zone|
|       111|         111|              4|     Brooklyn| Green-Wood Cemetery|   Boro Zone|
|       109|         109|              4|Staten Island|         Great Kills|   Boro Zone|
|       204|         204|              4|Staten Island|   Rossville/Woodrow|   Boro Zone|
|       199|         199|              4|        Bronx|       Rikers Island|   Boro Zone|
|         